# CSR Crawler
Crawl web dong (Playwright)

In [2]:
from google.colab import drive
import os
drive.mount('/content/drive')
WORK_DIR = '/content/drive/MyDrive/Crawl_Data/CrawlData'
DOWNLOAD_DIR = os.path.join(WORK_DIR, 'downloaded_files4')
os.makedirs(DOWNLOAD_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print('Work:', WORK_DIR)

Mounted at /content/drive
Work: /content/drive/MyDrive/Crawl_Data/CrawlData


In [ ]:
!pip install playwright beautifulsoup4 lxml aiohttp nest_asyncio -q
!playwright install chromium
!playwright install-deps  # Fix error: libatk-1.0.so.0 missing

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 MB 23.6 MB/s eta 0:00:00
(node:1031) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
164.7 MiB [] 0% 339.3s164.7 MiB [] 0% 71.2s164.7 MiB [] 0% 37.1s164.7 MiB [] 0% 27.4s164.7 MiB [] 0% 22.9s164.7 MiB [] 0% 20.8s164.7 MiB [] 0% 14.4s164.7 MiB [] 1% 9.2s164.7 MiB [] 1% 7.1s164.7 MiB [] 2% 6.0s164.7 MiB [] 3% 5.5s164.7 MiB [] 3% 5.2s164.7 MiB [] 3% 5.1s164.7 MiB [] 4% 5.7s164.7 MiB [] 4% 6.0s164.7 MiB [] 4% 5.4s164.7 MiB [] 5% 5.1s164.7 MiB [] 6% 4.7s164.7 MiB [] 6% 4.4s164.7 MiB [] 7% 4.1s164.7 MiB [] 8% 4.0s164.7 MiB [] 8% 3.8s164.7 MiB [] 9% 3.7s164.7 MiB [] 10% 3.6s164.7 MiB [] 10% 3.5s164.7 MiB [] 11% 3.4s164.7 MiB [] 12% 3.2s164.7 MiB [] 13% 3.0s164.7 MiB [] 14% 2.9s164.7 MiB [] 15% 2.9s164.7 

In [ ]:
import asyncio, re, json, aiohttp, nest_asyncio
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, unquote
from datetime import datetime
nest_asyncio.apply()

INPUT_FILE = os.path.join(WORK_DIR, 'csr_urls.json')
OUTPUT_FILE = os.path.join(WORK_DIR, 'output_csr.json')
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

In [ ]:
def get_filename(url):
    path = urlparse(url).path
    filename = unquote(os.path.basename(path))
    if not filename or filename == '/': filename = f'file_{hash(url) % 100000}'
    if '.' not in filename: filename = filename + '.bin'
    return filename

def is_downloadable(url, exts): return any(urlparse(url.lower()).path.endswith(e) for e in exts)
def is_same_domain(url, base): return urlparse(base).netloc == urlparse(url).netloc

def extract_links(html, base, selector=None, exts=None, require_ext=False):
    soup = BeautifulSoup(html, 'lxml')
    links, seen = [], set()
    if selector and selector.get('selector') and selector.get('type') == 'css':
        for elem in soup.select(selector['selector']):
            if elem.name == 'a' and elem.get('href'):
                url = urljoin(base, elem['href'])
                if url not in seen: seen.add(url); links.append(url)
            for a in elem.find_all('a', href=True):
                url = urljoin(base, a['href'])
                if url not in seen: seen.add(url); links.append(url)
    else:
        for a in soup.find_all('a', href=True):
            url = urljoin(base, a['href'])
            if url not in seen:
                if require_ext and exts:
                    if is_downloadable(url, exts): seen.add(url); links.append(url)
                else: seen.add(url); links.append(url)
    return links

In [ ]:
async def download(session, url, folder, fname):
    r = {'url': url, 'filename': fname, 'status': 'pending'}
    try:
        # Kiểm tra file đã tồn tại
        path = os.path.join(folder, fname)
        if os.path.exists(path):
            r['status'] = 'skipped'
            r['reason'] = 'already_exists'
            r['path'] = path
            r['size'] = os.path.getsize(path)
            return r

        async with session.get(url, timeout=aiohttp.ClientTimeout(120)) as resp:
            if resp.status == 200:
                if 'Content-Disposition' in resp.headers:
                    cd = resp.headers['Content-Disposition']
                    for p in [r"filename\*=UTF-8''(.+)", r'filename="(.+)"', r"filename='(.+)'", r'filename=([^;\s]+)']:
                        m = re.search(p, cd)
                        if m: r['filename'] = unquote(m.group(1).strip()); break

                # Kiểm tra lại với filename mới từ Content-Disposition
                path = os.path.join(folder, r['filename'])
                if os.path.exists(path):
                    r['status'] = 'skipped'
                    r['reason'] = 'already_exists'
                    r['path'] = path
                    r['size'] = os.path.getsize(path)
                    return r

                c = 1; b, ext = os.path.splitext(path)
                while os.path.exists(path): path = f'{b}_{c}{ext}'; c += 1
                content = await resp.read()
                with open(path, 'wb') as f: f.write(content)
                r['status'], r['size'], r['path'] = 'success', len(content), path
            else: r['status'], r['error'] = 'failed', f'HTTP {resp.status}'
    except Exception as e: r['status'], r['error'] = 'failed', str(e)
    return r

In [ ]:
async def crawl_multi_level(cfg, options):
    url = cfg.get('url')
    levels = cfg.get('levels', [])
    exts = cfg.get('file_extensions', ['.pdf'])
    delay = options.get('delay_between_requests', 1)
    max_files = options.get('max_files', 0)
    result = {'url': url, 'levels': [], 'files': [], 'status': 'pending'}
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        try:
            current_urls = [url]
            for level_idx, level_cfg in enumerate(levels):
                level_name = level_cfg.get('name', f'Level {level_idx+1}')
                is_download = level_cfg.get('is_download', False)
                selector = level_cfg.get('selector')
                max_pages = 100000
                max_pagination = 100000
                print(f'\n  Level {level_idx+1}: {level_name} ({len(current_urls)} urls)')
                all_links, crawled = [], []

                for i, u in enumerate(current_urls[:max_pages], 1):
                    print(f'  [{i}/{min(len(current_urls), max_pages)}] {u[:55]}...')
                    try:
                        await asyncio.sleep(delay)
                        await page.goto(u, wait_until='networkidle', timeout=60000)
                        await page.wait_for_timeout(2000)

                        # Xử lý phân trang
                        page_num = 1
                        while page_num <= max_pagination:
                            html = await page.content()
                            links = extract_links(html, u, selector, exts, require_ext=is_download and not selector)
                            if not is_download:
                                links = [l for l in links if is_same_domain(l, url) and l != u]
                            print(f'      -> Page {page_num}: {len(links)} links')

                            # Nếu là download level, tải ngay
                            if is_download and links:
                                if max_files > 0 and len(result['files']) >= max_files:
                                    print(f'      -> Reached max_files limit ({max_files})')
                                    break
                                remaining = max_files - len(result['files']) if max_files > 0 else len(links)
                                to_download = links[:remaining]
                                print(f'      -> Downloading {len(to_download)} files...')
                                async with aiohttp.ClientSession(headers=HEADERS) as s:
                                    tasks = [download(s, link, DOWNLOAD_DIR, get_filename(link)) for link in to_download]
                                    downloaded = await asyncio.gather(*tasks)
                                    result['files'].extend(downloaded)
                                    for dl in downloaded:
                                        if dl['status'] == 'success':
                                            status = 'OK'
                                        elif dl['status'] == 'skipped':
                                            status = 'SKIP'
                                        else:
                                            status = 'FAIL'
                                        print(f'        [{status}] {dl["filename"][:40]}')
                            else:
                                all_links.extend(links)

                            # Tìm trang tiếp theo
                            try:
                                soup = BeautifulSoup(html, 'lxml')
                                active_span = soup.find('span', class_='active')
                                if active_span:
                                    current_page_link = active_span.find('a')
                                    if current_page_link and 'submitsearch' in current_page_link.get('href', ''):
                                        match = re.search(r'submitsearch\((\d+)\)', current_page_link['href'])
                                        if match:
                                            current_page_num = int(match.group(1))
                                            next_page_num = current_page_num + 1
                                            print(f'      -> Current: page {current_page_num}, trying page {next_page_num}')

                                            next_selector = f"span:not(.active) > a[href*='submitsearch({next_page_num})']"
                                            next_btn = page.locator(next_selector)
                                            if await next_btn.count() > 0:
                                                await next_btn.click()
                                                await page.wait_for_timeout(3000)
                                                page_num += 1
                                            else:
                                                print(f'      -> No next page found')
                                                break
                                        else:
                                            break
                                    else:
                                        break
                                else:
                                    next_selector = "span:not(.active) > a[href*='submitsearch(2)']"
                                    next_btn = page.locator(next_selector)
                                    if await next_btn.count() > 0:
                                        await next_btn.click()
                                        await page.wait_for_timeout(3000)
                                        page_num += 1
                                    else:
                                        break
                            except Exception as e:
                                print(f'      -> Pagination error: {str(e)[:40]}')
                                break

                            # Kiểm tra max_files
                            if is_download and max_files > 0 and len(result['files']) >= max_files:
                                print(f'      -> Stopped: reached max_files ({max_files})')
                                break

                        crawled.append({'url': u, 'links_found': len(links), 'status': 'success'})
                    except Exception as e:
                        print(f'      Error: {str(e)[:40]}')
                        crawled.append({'url': u, 'status': 'failed', 'error': str(e)})

                if is_download:
                    result['levels'].append({'name': level_name, 'pages_crawled': len(crawled), 'total_downloaded': len(result['files'])})
                    print(f'\n  Total downloaded: {len(result["files"])}')
                    break
                else:
                    seen = set()
                    unique = [l for l in all_links if l not in seen and not seen.add(l)]
                    result['levels'].append({'name': level_name, 'pages_crawled': len(crawled), 'links_found': len(unique)})
                    print(f'  Total: {len(unique)} links')
                    current_urls = unique
                    if not current_urls: print('  No more URLs'); break

            result['status'] = 'success'
        except Exception as e: result['status'], result['error'] = 'failed', str(e)
        finally: await browser.close()
    return result

In [ ]:
async def main():
    with open(INPUT_FILE, 'r', encoding='utf-8') as f: data = json.load(f)
    urls, opts = data.get('urls', []), data.get('options', {})
    print(f'URLs: {len(urls)}')
    if opts.get('max_files'): print(f'Max files: {opts["max_files"]}')
    print('='*50)
    results = []
    for i, cfg in enumerate(urls, 1):
        url, mode = cfg.get('url'), cfg.get('crawl_mode', 'one_level')
        print(f'\n[{i}] {url[:50]}...')
        print(f'  Mode: {mode}')
        if mode == 'multi_level': r = await crawl_multi_level(cfg, opts)
        elif mode == 'two_level': r = await crawl_two_level(cfg, opts)
        else: r = await crawl_one_level(cfg, opts)
        ok = sum(1 for f in r['files'] if f['status']=='success')
        print(f'\n  Downloaded: {ok}/{len(r["files"])}')
        results.append(r)
    return results

all_results = asyncio.get_event_loop().run_until_complete(main())
print('\nDone!')

URLs: 10

[1] https://bqp.vn/home/vbpl?field=/Mod/sa-mod-site/sa...
  Mode: multi_level

  Level 1: Trang chi tiết (1 urls)
  [1/1] https://bqp.vn/home/vbpl?field=/Mod/sa-mod-site/sa-qlcd...
      -> Page 1: 20 links
      -> Current: page 1, trying page 2
      -> Page 2: 7 links
      -> Current: page 2, trying page 3
      -> No next page found
  Total: 27 links

  Level 2: Links download (27 urls)
  [1/27] https://bqp.vn/vn/van-ban/sa-qlcddh-vbpl-htvb/sa-qlcddh...
      -> Page 1: 1 links
      -> Downloading 1 files...
        [SKIP] QD6375BQP.pdf
  [2/27] https://bqp.vn/vn/van-ban/sa-qlcddh-vbpl-htvb/sa-qlcddh...
      -> Page 1: 1 links
      -> Downloading 1 files...
        [SKIP] VBHN76BQP.pdf
  [3/27] https://bqp.vn/vn/van-ban/sa-qlcddh-vbpl-htvb/sa-qlcddh...
      -> Page 1: 1 links
      -> Downloading 1 files...
        [SKIP] VBHN75BQP.pdf
  [4/27] https://bqp.vn/vn/van-ban/sa-qlcddh-vbpl-htvb/sa-qlcddh...
      -> Page 1: 1 links
      -> Downloading 1 files...
        

In [ ]:


output = {'results': all_results, 'summary': {'urls': len(all_results), 'total_files': sum(len(r['files']) for r in all_results), 'downloaded': sum(sum(1 for f in r['files'] if f['status']=='success') for r in all_results)}, 'crawled_at': datetime.now().isoformat()}
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f: json.dump(output, f, indent=2, ensure_ascii=False)
print(f'Total: {output["summary"]["total_files"]}')
print(f'Downloaded: {output["summary"]["downloaded"]}')

Total: 1050
Downloaded: 0
